# neo4j 入库 pipeline

用来实验论文知识图（`docs/designs/graph_schema.md`）逐篇入库、更新、扩展的流程。

测试数据是 `_scratch/data_model_examples/` 下人工伪造的三批 yml，按真实顺序**一批一批**进库，
而不是一次性全部加载：先有 01 BERT；02 RAG 入库时要挂到 BERT 的既有节点上；03 GraphRAG 再挂到 RAG 上。
每批都先和库里现有内容对照、看清楚会改什么，再写入：

    yml 批次 ──load_batch──▶ 本批 nodes / relationships（批内校验：label 在 schema 内、id 不重复）
             ──plan_batch──▶ 对照库：节点、关系各分 新建 / 更新 / 不变；
                              关系引用的既有节点；两头都找不到的悬空引用 → 直接报错，不写
             ──apply_plan──▶ 一个写事务，按 id MERGE（属性 SET +=：只增改、不删）
             ──复核────────▶ 再 plan 一次，必须全部"不变"

流程会多次重跑：已入库且内容没变的批次 plan 出来全是"不变"，直接跳过写入。

## step1 预设参数

In [ ]:
import os
from pathlib import Path

DATA = Path.cwd() / "_scratch/data_model_examples"   # 三批测试数据，按入库顺序编号

# ── 连接参数：默认值对应 infra/neo4j/docker-compose.yml ──
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

RESET = False   # True：先删掉这些测试数据涉及的节点（按 yml 里的 id，不动库里其他内容），从空图重演

## step2 连接 neo4j，看库里已有什么

- 每个 label 建 `id` 唯一约束（`IF NOT EXISTS`，重跑无副作用），保证同一个 id 不会被建出两份；
- `library_status()` 打印当前库的内容，每批入库后都会再调一次。

In [ ]:
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import pandas as pd
import yaml
from neo4j import GraphDatabase, RoutingControl

# 空库时查 p.title 之类会收到“属性键不存在”的提示（UNRECOGNIZED 类），关掉这一类
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_disabled_classifications=["UNRECOGNIZED"])
driver.verify_connectivity()
print(NEO4J_URI, driver.get_server_info().agent)

def q(cypher: str, **params) -> list[dict]:
    """只读查询的简写。"""
    records, _, _ = driver.execute_query(cypher, params, database_=NEO4J_DB, routing_=RoutingControl.READ)
    return [r.data() for r in records]

# graph_schema.md 里的 12 种 Node
LABELS = ["Paper", "Contribution", "Method", "MethodConcept", "Claim", "ClaimConcept",
          "Resource", "ResourceRecord", "Experiment", "Metric", "Condition", "ContentUnit"]
for label in LABELS:
    driver.execute_query(f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS "
                         f"FOR (n:{label}) REQUIRE n.id IS UNIQUE", database_=NEO4J_DB)

def library_status():
    labels = q("MATCH (n) UNWIND labels(n) AS l RETURN l, count(*) AS c ORDER BY l")
    rels = q("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
    papers = q("MATCH (p:Paper) RETURN p.id AS id, p.title AS title, p.markdown_path IS NOT NULL AS full "
               "ORDER BY p.year, p.id")
    print(f"库内：节点 {sum(x['c'] for x in labels)}，关系 {rels}")
    print("  " + ", ".join(f"{x['l']} {x['c']}" for x in labels) if labels else "  （空库）")
    for p in papers:
        print(f"  {'全文' if p['full'] else '仅标题'}  {p['id']:<16} {p['title'][:70]}")

if RESET:
    ids = [n["id"] for f in sorted(DATA.glob("*.yml"))
           for n in yaml.safe_load(f.read_text(encoding="utf-8"))["nodes"]]
    deleted = driver.execute_query("MATCH (n) WHERE n.id IN $ids DETACH DELETE n", ids=ids,
                                   database_=NEO4J_DB).summary.counters.nodes_deleted
    print(f"RESET：删除 {deleted} 个节点")

library_status()

## step3 入库的三个步骤

- **`load_batch`**：读 yml，只做和库无关的检查。节点 id 同时写进属性 `id`；label 和关系类型要拼进 Cypher 字符串，只放行 schema 内的 label 和标识符形式的关系类型。
- **`plan_batch`**：拿本批涉及的全部 id 去库里查一遍，分类：
  - 节点：库里没有 → 新建；有但 yml 里某个属性值不同 → 更新（列出字段级差异）；否则不变。同一 id 在库里是别的 label → 报错。
  - 关系以 (from, type, to) 为身份，同样分新建 / 更新 / 不变。`DIFFERS_FROM` 等语义对称关系不分方向，库里有反向的一条也算已存在。
  - 本批只引用、不定义的 id 必须已在库里（前面批次入库的），否则是悬空引用，整批拒绝。
- **`apply_plan`**：只写新建和更新的部分，全部放进一个写事务；写完再 plan 一次，必须全部"不变"。

更新语义是 `SET n += props`：yml 里出现的属性覆盖，yml 里没写的属性保留。所以"先建仅有标题的 Paper、后补全"
这种增量补充直接成立；反过来，从 yml 里删掉一个属性**不会**删掉库里的值。

In [ ]:
SYMMETRIC = {"DIFFERS_FROM", "OVERLAPS_WITH", "CONTRADICTS"}   # graph_schema.md 里的语义对称关系
REL_TYPE_RE = re.compile(r"^[A-Z][A-Z0-9_]*$")

def check_batch(raw: dict, name: str) -> dict:
    nodes, rels = {}, {}
    for n in raw.get("nodes") or []:
        assert n["id"] not in nodes, f"{name}: 批内重复定义 {n['id']}"
        bad = [l for l in n["labels"] if l not in LABELS]
        assert not bad, f"{name}: {n['id']} 的 label {bad} 不在 schema 里"
        nodes[n["id"]] = {"labels": n["labels"], "props": {"id": n["id"], **(n.get("properties") or {})}}
    for r in raw.get("relationships") or []:
        key = (r["from"], r["type"], r["to"])
        assert REL_TYPE_RE.match(r["type"]), f"{name}: 关系类型不合法 {r['type']!r}"
        assert key not in rels, f"{name}: 批内重复关系 {key}"
        rels[key] = r.get("properties") or {}
    return {"name": name, "nodes": nodes, "rels": rels}

def load_batch(path: Path) -> dict:
    return check_batch(yaml.safe_load(path.read_text(encoding="utf-8")), path.name)

def diff_props(old: dict, new: dict) -> dict:
    """yml 里写了、且和库里不同的字段：{key: (库里的值, yml 的值)}。"""
    return {k: (old.get(k), v) for k, v in new.items() if old.get(k) != v}

In [ ]:
@dataclass
class Plan:
    batch: dict
    new_nodes: list = field(default_factory=list)
    upd_nodes: dict = field(default_factory=dict)    # id -> diff
    same_nodes: list = field(default_factory=list)
    referenced: dict = field(default_factory=dict)   # 本批只引用的既有节点 id -> labels
    new_rels: list = field(default_factory=list)
    upd_rels: dict = field(default_factory=dict)     # key -> diff
    same_rels: list = field(default_factory=list)

    @property
    def empty(self) -> bool:
        return not (self.new_nodes or self.upd_nodes or self.new_rels or self.upd_rels)

def plan_batch(b: dict) -> Plan:
    p = Plan(b)
    endpoints = {x for a, _, c in b["rels"] for x in (a, c)}
    # 测试规模小，这里按 id 全图查；规模大了应带上 label，走唯一约束的索引
    db = {r["id"]: r for r in q("MATCH (n) WHERE n.id IN $ids "
                                "RETURN n.id AS id, labels(n) AS labels, properties(n) AS props",
                                ids=sorted(set(b["nodes"]) | endpoints))}

    for nid, n in b["nodes"].items():
        if nid not in db:
            p.new_nodes.append(nid)
            continue
        assert set(db[nid]["labels"]) == set(n["labels"]), \
            f"{b['name']}: {nid} 在库里是 {db[nid]['labels']}，本批写成 {n['labels']}"
        d = diff_props(db[nid]["props"], n["props"])
        (p.upd_nodes.__setitem__(nid, d) if d else p.same_nodes.append(nid))

    outside = endpoints - set(b["nodes"])
    dangling = sorted(outside - set(db))
    assert not dangling, (f"{b['name']}: 关系引用了库里没有、本批也没定义的节点 {dangling}"
                          "——前面的批次是不是还没入库？")
    p.referenced = {i: db[i]["labels"] for i in sorted(outside)}

    keys = [{"from": a, "type": t, "to": c} for a, t, c in b["rels"]]
    existing = {(r["from"], r["type"], r["to"]): r["props"] for r in q(
        """UNWIND $keys AS k
           MATCH (a {id: k.from})-[r]-(c {id: k.to})
           WHERE type(r) = k.type AND (startNode(r) = a OR k.type IN $sym)
           RETURN k.from AS from, k.type AS type, k.to AS to, properties(r) AS props""",
        keys=keys, sym=sorted(SYMMETRIC))}
    for key, props in b["rels"].items():
        if key not in existing:
            p.new_rels.append(key)
        elif d := diff_props(existing[key], props):
            p.upd_rels[key] = d
        else:
            p.same_rels.append(key)
    return p

def short(v, n=60):
    s = str(v).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "…"

def show_plan(p: Plan):
    b = p.batch
    print(f"== {b['name']} ==")
    print(f"节点  新建 {len(p.new_nodes)} · 更新 {len(p.upd_nodes)} · 不变 {len(p.same_nodes)}")
    print(f"关系  新建 {len(p.new_rels)} · 更新 {len(p.upd_rels)} · 不变 {len(p.same_rels)}")
    if p.new_nodes:
        c = Counter(b["nodes"][i]["labels"][0] for i in p.new_nodes)
        print("  新建节点：" + ", ".join(f"{k} {v}" for k, v in c.items()))
    if p.referenced:
        print(f"引用既有节点 {len(p.referenced)} 个（前面批次已入库，本批只往上挂关系）：")
        for i, labels in p.referenced.items():
            hooks = [f"{'←' if c == i else '→'} {t} {a if c == i else c}" for a, t, c in p.new_rels if i in (a, c)]
            print(f"  {i} {labels}  " + ("; ".join(hooks) if hooks else "（关系已存在）"))
    for i, d in p.upd_nodes.items():
        print(f"更新节点 {i}：")
        for k, (old, new) in d.items():
            print(f"    {k}: {short(old)} → {short(new)}")
    for key, d in p.upd_rels.items():
        print(f"更新关系 {key}：" + "; ".join(f"{k}: {short(o)} → {short(n)}" for k, (o, n) in d.items()))
    if p.empty:
        print("→ 与库内容一致，无需写入")

In [ ]:
def apply_plan(p: Plan):
    b = p.batch
    if p.empty:
        print(f"{b['name']}：无变化，跳过写入")
        return

    def work(tx):
        # 节点：按 label 组合分组，每组一条 UNWIND + MERGE
        groups = defaultdict(list)
        for nid in p.new_nodes + list(p.upd_nodes):
            n = b["nodes"][nid]
            groups[tuple(n["labels"])].append({"id": nid, "props": n["props"]})
        for labels, rows in groups.items():
            extra = "".join(f":{l}" for l in labels[1:])
            c = tx.run(f"UNWIND $rows AS row MERGE (n:{labels[0]} {{id: row.id}}) SET n += row.props"
                       + (f" SET n{extra}" if extra else "") + " RETURN count(n) AS c", rows=rows).single()["c"]
            assert c == len(rows)
        # 关系：按类型分组；语义对称的用无方向 MERGE，已有反向的一条就复用它
        groups = defaultdict(list)
        for key in p.new_rels + list(p.upd_rels):
            groups[key[1]].append({"from": key[0], "to": key[2], "props": b["rels"][key]})
        for t, rows in groups.items():
            arrow = "-" if t in SYMMETRIC else "->"
            c = tx.run(f"UNWIND $rows AS row MATCH (a {{id: row.from}}) MATCH (c {{id: row.to}}) "
                       f"MERGE (a)-[r:{t}]{arrow}(c) SET r += row.props RETURN count(r) AS c", rows=rows).single()["c"]
            assert c == len(rows), f"{t}: {len(rows)} 条里只写入 {c} 条（端点没匹配上）"

    with driver.session(database=NEO4J_DB) as s:
        s.execute_write(work)   # 单事务：中途出错整批回滚，不会留下半批
    print(f"{b['name']}：写入 节点 {len(p.new_nodes) + len(p.upd_nodes)}，关系 {len(p.new_rels) + len(p.upd_rels)}")

    again = plan_batch(b)   # 复核：写完之后库内容应与本批完全一致
    assert again.empty, "复核失败：写入后仍有差异"
    print("  ✓ 复核通过")

## step4 第 1 批：BERT

图谱的起点。本批没有引用任何外部节点；第一次运行时应全部是"新建"，之后重跑全部是"不变"。

In [ ]:
b01 = load_batch(DATA / "01_bert.yml")
p01 = plan_batch(b01)
show_plan(p01)

In [ ]:
apply_plan(p01)
library_status()

## step5 第 2 批：RAG

在 BERT 的基础上扩展。plan 里要看三件事：

- **挂接既有节点**：`paper:bert`（被 CITES）、`method:bert`（被 HAS_METHOD reused）、`resource:bert-checkpoints`（被 RELATES_TO、DESCRIBES）
  不在本批里定义，只被关系引用。它们的属性不动，但从此多了来自 RAG 的边；
- **仅有标题的 Paper**：`paper:dpr` 只带 title 入库，后面补全（见 step7）；
- **第一个跨论文命题** `cc:external-memory-helps`。

如果 01 还没入库就跑这一步，plan 会因为悬空引用直接报错。

In [ ]:
b02 = load_batch(DATA / "02_rag.yml")
p02 = plan_batch(b02)
show_plan(p02)

In [ ]:
apply_plan(p02)
library_status()

## step6 第 3 批：GraphRAG

大量挂接 02 的既有节点：`paper:rag`（CITES）、`method:rag-2020`（DIFFERS_FROM，语义对称）、`concept:rag`
（SUBTYPE_OF、INSTANCE_OF）、`claim:rag:retrieval-helps`（被 QUALIFIES）、`cc:external-memory-helps`（被 REFINES）。

In [ ]:
b03 = load_batch(DATA / "03_graphrag.yml")
p03 = plan_batch(b03)
show_plan(p03)

In [ ]:
apply_plan(p03)
library_status()

## step7 更新既有节点：补全 `paper:dpr`

前三批只有新增，没有改动已有节点的属性。这里在 notebook 里写一小批补丁（**不是**测试数据目录里的文件），演示后来拿到
DPR 的元数据、回头补全仅有标题的 Paper：同一个 id，plan 里是"更新"并列出字段差异，title 不写也不会被清掉。

In [ ]:
patch_dpr = check_batch({
    "nodes": [{
        "id": "paper:dpr",
        "labels": ["Paper"],
        "properties": {"year": 2020, "arxiv_id": "2004.04906", "paper_type": "method"},
    }],
}, name="patch_dpr（inline）")
p_patch = plan_batch(patch_dpr)
show_plan(p_patch)

In [ ]:
apply_plan(p_patch)
q("MATCH (p:Paper {id: 'paper:dpr'}) RETURN properties(p) AS props")[0]["props"]

## step8 查看跨批次连起来的部分

三批各自入库，这里查的是只有放在一起才有的路径。

**命题细化**：03 的 Claim 所表达的命题 REFINES 02 的命题；03 的另一条 Claim 对 02 的 Claim 做 QUALIFIES。

In [ ]:
pd.DataFrame(q("""
MATCH (p2:Paper)-[:HAS_CLAIM]->(c2:Claim)-[:EXPRESSES]->(cc2:ClaimConcept)
      -[:REFINES]->(cc1:ClaimConcept)<-[:EXPRESSES]-(c1:Claim)<-[:HAS_CLAIM]-(p1:Paper)
RETURN p2.id AS paper, 'REFINES' AS rel, p1.id AS earlier_paper, cc2.text AS concept, cc1.text AS earlier_concept
UNION
MATCH (p2:Paper)-[:HAS_CLAIM]->(c2:Claim)-[:QUALIFIES]->(c1:Claim)<-[:HAS_CLAIM]-(p1:Paper)
RETURN p2.id AS paper, 'QUALIFIES' AS rel, p1.id AS earlier_paper, c2.text AS concept, c1.text AS earlier_concept
"""))

**多篇论文共用的方法**，以及方法所属类别（`INSTANCE_OF` / `SUBTYPE_OF`）。

In [ ]:
pd.DataFrame(q("""
MATCH (p:Paper)-[h:HAS_METHOD]->(m:Method)
WITH m, collect(p.id + ' (' + h.role + ')') AS papers
OPTIONAL MATCH (m)-[:INSTANCE_OF]->(c:MethodConcept)
OPTIONAL MATCH (c)-[:SUBTYPE_OF]->(sup:MethodConcept)
RETURN m.id AS method, papers, c.name AS concept, sup.name AS super_concept
ORDER BY size(papers) DESC, method
"""))

In [ ]:
library_status()
driver.close()